# 10 · EDEN: a precipitation GeoTIFF with honest units

**Objective:** Read an EDEN-exported precipitation COG, transform an AOI into its CRS, crop only that window, and convert the declared precipitation convention.

**Route:** Manual acquisition / local COG · **Audience:** researcher / developer · **Duration:** 15–30 minutes after setup.

[Notebook index](README.md) · [Environment setup](../docs/setup/README.md) · [Accounts and API keys](../docs/setup/accounts.md) · [Data quality](../docs/reference/data-quality.md) · [Guide home](../README.md)

**Verification:** authored from the original workflow and official specifications; live access and your input files have **not been tested in this guide**. All distributed code cells have empty outputs. The release verification report records the separate offline checks. Run from a fresh kernel in numbered cell order.

**Expected result:** a cropped monthly precipitation map and explicit temporal-unit provenance. Empty searches or missing permissions are possible; they are not evidence of a scientific result.

[Service overview and access route](../docs/services/eden.md)


## 1. Load the local environment

Install the documented environment before opening this notebook. Paths in `.env` are relative to the repository root. Authentication is kept outside committed notebooks.


In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples" / "common.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside DestinE-partner-guide.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=False)
from examples.common import required_env, input_path, public_url, record_provenance


## 2. Export from EDEN and retain its metadata

Find EDEN through the [DestinE catalogue](https://platform.destine.eu/), select an ERA5-Land precipitation product and a modest spatial/temporal subset, and save the COG with its export metadata. Set `INPUT_EDEN_PRECIP`, `EDEN_YEAR`, `EDEN_MONTH`, and `EDEN_PRECIP_CONVENTION`. EDEN's adapter API is distinct from the DestinE Data Lake HDA STAC v2 API; this notebook reads a manual EDEN export.

For a monthly mean of daily accumulations in metres/day, choose `monthly_mean_daily_m`: monthly total = value × 1000 × days in month. For a documented monthly accumulated depth in metres, choose `accumulated_m`: total = value × 1000. The filename alone cannot establish which convention applies.


In [ ]:
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds
from examples.climate import precipitation_mm

path = input_path("INPUT_EDEN_PRECIP")
year, month = int(required_env("EDEN_YEAR")), int(required_env("EDEN_MONTH"))
convention = required_env("EDEN_PRECIP_CONVENTION")
if convention not in {"monthly_mean_daily_m", "accumulated_m"}:
    raise ValueError("Declare the export's temporal convention from its metadata.")
bbox = [22, 58, 28, 62]
with rasterio.open(path) as src:
    if src.crs is None:
        raise ValueError("Export has no CRS.")
    print({"crs": str(src.crs), "shape": src.shape, "bands": src.count, "units": src.units, "tags": src.tags()})
    if src.count != 1:
        raise ValueError("Choose one documented monthly precipitation band/export.")
    bounds = transform_bounds("EPSG:4326", src.crs, *bbox)
    window = from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
    window = window.intersection(rasterio.windows.Window(0, 0, src.width, src.height))
    if window.width * window.height > 4_000_000:
        raise ValueError("Use a smaller AOI for this example.")
    raw = src.read(1, window=window, masked=True).astype(float).filled(np.nan)
    # Honour GDAL scale/offset before converting physical units.
    values = raw * src.scales[0] + src.offsets[0]
    precip_mm = precipitation_mm(values, convention, year=year, month=month)
    grid_crs = str(src.crs)
assert np.isfinite(precip_mm).any()
pd.Series(precip_mm[np.isfinite(precip_mm)]).describe()


## 3. Show a monthly total only after validating the convention

The pixel mean below is a descriptive raster summary; it is not area-weighted and is not a country or catchment total.


In [ ]:
plt.figure(figsize=(7, 4))
plt.imshow(precip_mm, cmap="Blues")
plt.colorbar(label="Monthly precipitation (mm), declared export convention")
plt.title(f"EDEN export — {year}-{month:02d}, selected rectangle")
plt.show()
record_provenance("eden-precipitation.json", input_file=path.name, year=year, month=month, convention=convention, units="mm/month", bbox_wgs84=bbox, crs=grid_crs, valid_pixels=int(np.isfinite(precip_mm).sum()), statistics="unweighted pixel descriptions only")


## Interpret and verify

Verify exported variable units and any EDEN aggregation before selecting the convention. A map of monthly depth is not observed streamflow, groundwater, or water-table depth. Do not add monthly day counts to a product that is already a monthly total.

Check CRS, dates, masks, units, spatial extent, and valid sample count before comparing results. Keep product IDs and request parameters with the exported output. Do not treat synthetic/offline checks as verification of a remote service.


## Official references

- [ECMWF ERA5-Land temporal conventions](https://confluence.ecmwf.int/spaces/CKB/pages/140385202/ERA5-Land%2Bdata%2Bdocumentation)
- [Rasterio window reads](https://rasterio.readthedocs.io/en/stable/topics/windowed-rw.html)
- [EDEN official documentation](https://platform.destine.eu/docs/eden/doc/index.html)

[Return to notebook index](README.md) · [Continue through Start here](../docs/start-here.md) · [Guide home](../README.md)
